In [1]:
!pip install pandas numpy scikit-learn fairlearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.0/240.0 kB 4.2 MB/s eta 0:00:0000:01


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

seed = 42

def preprocess_bank_for_fair_bbn(csv_path='/kaggle/input/remaining2/bankmarketing/bank/bank-full.csv'):
    # -------------------------
    # Load dataset & clean
    # -------------------------
    df = pd.read_csv(csv_path, sep=';')
    df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    
    # Target variable
    df['y'] = np.where(df['y'] == 'yes', 1, 0)
    
    # -------------------------
    # Upsample positives
    # (preserve job + marital distribution)
    # -------------------------
    positives = df[df['y'] == 1]
    negatives = df[df['y'] == 0]
    
    dist = positives.groupby(['job', 'marital']).size().reset_index(name='count')
    dist['up_count'] = (dist['count'] * len(negatives) / dist['count'].sum()).round().astype(int)
    
    upsamples = []
    for _, row in dist.iterrows():
        group = positives[(positives['job'] == row['job']) & (positives['marital'] == row['marital'])]
        if len(group) > 0:
            upsamples.append(resample(group, replace=True, n_samples=row['up_count'], random_state=seed))
    
    df = pd.concat([negatives] + upsamples).sample(frac=1, random_state=seed).reset_index(drop=True)
    
    # -------------------------
    # Sensitive attributes
    # -------------------------
    marital_map = {"married": 0, "single": 1, "divorced": 2}
    df['marital_label'] = df['marital'].map(marital_map)
    
    job_map = {job: i for i, job in enumerate(df['job'].unique())}
    df['job_label'] = df['job'].map(job_map)
    
    # -------------------------
    # Define feature groups
    # -------------------------
    categorical_cols = ['job', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
    numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
    
    # -------------------------
    # Preprocessor for NN
    # -------------------------
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    
    # -------------------------
    # BBN preprocessing (discretization)
    # -------------------------
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['marital', 'job']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    # -------------------------
    # Train/test split
    # -------------------------
    X = df.drop(columns=['y', 'marital_label', 'job_label'])
    y = df['y'].values
    sens_marital = df['marital_label']
    sens_job = df['job_label']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_marital_train, sens_marital_test, sens_job_train, sens_job_test = \
        train_test_split(
            X, y, sens_marital, sens_job, test_size=0.2, stratify=y, random_state=seed
        )
    
    # Apply NN preprocessing
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    # Align BBN dataset
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_marital_train': sens_marital_train.reset_index(drop=True),
        'sens_marital_test': sens_marital_test.reset_index(drop=True),
        'sens_job_train': sens_job_train.reset_index(drop=True),
        'sens_job_test': sens_job_test.reset_index(drop=True)
    }

# Run preprocessing
bank_data = preprocess_bank_for_fair_bbn('/kaggle/input/remaining2/bankmarketing/bank/bank-full.csv')
print("Bank Marketing NN shape:", bank_data['X_train_nn'].shape)
print("Bank Marketing BBN shape:", bank_data['bbn_train_df'].shape)


Bank Marketing NN shape: (9692, 44)
Bank Marketing BBN shape: (9692, 19)


In [5]:
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
# bbn_adapter_bank.py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

# seed should match your preprocessing file
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# ---------------- SimpleFairBBN (adapted) ----------------
class SimpleFairBBN:
    def __init__(self, feature_names=None, target_name='y'):
        self.feature_names = feature_names or []
        self.target_name = target_name
        self.model = None
        self.inference = None

    def fit(self, df_discrete, y, include_sensitive=True):
        # df_discrete: discretized dataframe (categorical codes)
        data = df_discrete.copy().reset_index(drop=True)
        data[self.target_name] = y
        # choose top 6 features + optionally sensitive attrs if present
        candidate_features = list(df_discrete.columns)
        selected = candidate_features[:6]
        if include_sensitive:
            if 'marital' in df_discrete.columns and 'marital' not in selected:
                selected.append('marital')
            if 'job' in df_discrete.columns and 'job' not in selected:
                selected.append('job')
        edges = [(f, self.target_name) for f in selected]
        self.feature_names = selected
        self.model = DiscreteBayesianNetwork(edges)
        # Fit with BayesianEstimator BDeu smoothing
        self.model.fit(data, estimator=BayesianEstimator, prior_type='BDeu', equivalent_sample_size=5)
        self.inference = VariableElimination(self.model)

    def predict_proba(self, df_discrete):
        Xdf = df_discrete.reset_index(drop=True)
        probs = []
        for _, row in Xdf.iterrows():
            evidence = {}
            used = 0
            for f in self.feature_names:
                if f in row and not pd.isna(row[f]) and used < 3:
                    try:
                        evidence[f] = int(row[f])
                        used += 1
                    except Exception:
                        continue
            try:
                if evidence:
                    q = self.inference.query(variables=[self.target_name], evidence=evidence)
                else:
                    q = self.inference.query(variables=[self.target_name])
                # q.values index: assume binary 0/1 target; if not, fallback 0.5
                p1 = q.values[1] if len(q.values) == 2 else 0.5
            except Exception:
                p1 = 0.5
            probs.append(p1)
        return np.array(probs)

# ---------------- Adapter + Adversary ----------------
class AdapterNN(nn.Module):
    def __init__(self, in_dim=3, hidden_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.out = nn.Linear(hidden_dim // 2, 1)

    def forward(self, x, return_features=False):
        h = self.act(self.fc1(x))
        h2 = self.act(self.fc2(h))
        logit = self.out(h2)
        if return_features:
            return logit, h2
        return logit

class AdversaryNN(nn.Module):
    def __init__(self, in_dim, marital_classes, job_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.marital_head = nn.Linear(32, marital_classes)
        self.job_head = nn.Linear(32, job_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.marital_head(s), self.job_head(s)

# ---------------- Training function (bank-adapted) ----------------
def train_fair_bbn_adapter_bank(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3):
    # Unpack preprocessed bank data (matches your preprocess_bank_for_fair_bbn output)
    X_train_nn = data_dict['X_train_nn']
    X_test_nn = data_dict['X_test_nn']
    bbn_train_df = data_dict['bbn_train_df']
    bbn_test_df = data_dict['bbn_test_df']
    y_train = data_dict['y_train']
    y_test = data_dict['y_test']
    sens_marital_train = data_dict['sens_marital_train']
    sens_marital_test = data_dict['sens_marital_test']
    sens_job_train = data_dict['sens_job_train']
    sens_job_test = data_dict['sens_job_test']

    # Baseline MLP (sklearn)
    print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=seed)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)
    base_marital_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_marital_test)
    base_marital_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_marital_test)
    base_job_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_job_test)
    base_job_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_job_test)
    print(f"Baseline MLP Accuracy: {base_acc:.4f}")

    # Train BBN on discretized bank data
    print("Training fair BBN...")
    bbn = SimpleFairBBN(feature_names=list(bbn_train_df.columns), target_name='y')
    bbn.fit(bbn_train_df, y_train, include_sensitive=True)

    # Create adapter inputs: P(y|all), P(y|marital), P(y|job)
    p_all = bbn.predict_proba(bbn_train_df).reshape(-1, 1)
    p_marital = bbn.predict_proba(bbn_train_df[['marital']]).reshape(-1, 1)
    p_job = bbn.predict_proba(bbn_train_df[['job']]).reshape(-1, 1)
    adapter_in_train = torch.tensor(np.hstack([p_all, p_marital, p_job]), dtype=torch.float32)

    p_all_test = bbn.predict_proba(bbn_test_df).reshape(-1, 1)
    p_marital_test = bbn.predict_proba(bbn_test_df[['marital']]).reshape(-1, 1)
    p_job_test = bbn.predict_proba(bbn_test_df[['job']]).reshape(-1, 1)
    adapter_in_test = torch.tensor(np.hstack([p_all_test, p_marital_test, p_job_test]), dtype=torch.float32)

    # Targets -> tensors
    y_train_t = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    y_test_t = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)
    marital_train_t = torch.tensor(sens_marital_train.values.astype(int), dtype=torch.long)
    job_train_t = torch.tensor(sens_job_train.values.astype(int), dtype=torch.long)

    train_loader = DataLoader(TensorDataset(adapter_in_train, y_train_t, marital_train_t, job_train_t),
                              batch_size=batch_size, shuffle=True)

    # Initialize adapter & adversary
    adapter = AdapterNN(in_dim=3, hidden_dim=64)
    adversary = AdversaryNN(in_dim=32, marital_classes=len(sens_marital_train.unique()), job_classes=len(sens_job_train.unique()))
    adapter_opt = optim.Adam(adapter.parameters(), lr=lr)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

    print("Training adapter with adversarial + BBN fairness...")
    for epoch in range(epochs):
        adapter.train(); adversary.train()
        total_adapter_loss = 0.0; total_adv_loss = 0.0

        for batch in train_loader:
            batch_in, batch_y, batch_marital, batch_job = batch

            # --- Train adversary (on adapter features) ---
            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            m_logits, j_logits = adversary(features)
            adv_loss = (adv_loss_fn(m_logits, batch_marital) + adv_loss_fn(j_logits, batch_job)) / 2.0
            adv_loss.backward(); adv_opt.step(); total_adv_loss += adv_loss.item()

            # --- Train adapter (predict y and try to fool adversary) ---
            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit, batch_y)
            m_logits2, j_logits2 = adversary(features)
            adv_loss_for_adapter = (adv_loss_fn(m_logits2, batch_marital) + adv_loss_fn(j_logits2, batch_job)) / 2.0

            # BBN fairness regularization: Demographic parity penalty on adapter features
            # compute feature means by marital group if possible (safe fallback if group not present)
            try:
                grp0_mean = features[batch_marital == 0].mean(dim=0)
                grp1_mean = features[batch_marital == 1].mean(dim=0)
                dp_penalty = torch.abs(grp0_mean - grp1_mean).mean()
            except Exception:
                dp_penalty = torch.tensor(0.0)

            total_loss = pred_loss - lambda_adv * adv_loss_for_adapter + alpha_bbn * dp_penalty
            total_loss.backward()
            adapter_opt.step()
            total_adapter_loss += total_loss.item()

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:3d} | Adv Loss: {total_adv_loss / len(train_loader):.4f} | Adapter Loss: {total_adapter_loss / len(train_loader):.4f}")

    # ---------------- Evaluation ----------------
    adapter.eval(); adversary.eval()
    with torch.no_grad():
        test_logit, _ = adapter(adapter_in_test, return_features=True)
        test_probs = torch.sigmoid(test_logit.cpu()).numpy().flatten()
        test_pred = (test_probs > 0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_marital_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_marital_test)
    adv_marital_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_marital_test)
    adv_job_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_job_test)
    adv_job_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_job_test)

    # Print comparison
    print("\nBASELINE MLP RESULTS:")
    print(f" Accuracy: {base_acc:.4f}")
    print(f" Marital DP: {base_marital_dp:.4f}, Marital EO: {base_marital_eo:.4f}")
    print(f" Job     DP: {base_job_dp:.4f}, Job     EO: {base_job_eo:.4f}")

    print("\nBBN + Adapter (Adversarial + Fairness) RESULTS:")
    print(f" Accuracy: {adv_acc:.4f}")
    print(f" Marital DP: {adv_marital_dp:.4f}, Marital EO: {adv_marital_eo:.4f}")
    print(f" Job     DP: {adv_job_dp:.4f}, Job     EO: {adv_job_eo:.4f}")

    return {
        'baseline': {'pred': base_pred, 'acc': base_acc, 'marital_dp': base_marital_dp, 'marital_eo': base_marital_eo,
                     'job_dp': base_job_dp, 'job_eo': base_job_eo},
        'bbn_adapter': {'pred': test_pred, 'acc': adv_acc, 'marital_dp': adv_marital_dp, 'marital_eo': adv_marital_eo,
                        'job_dp': adv_job_dp, 'job_eo': adv_job_eo}
    }

# ---------------- Run (example) ----------------
if __name__ == '__main__':
    # Assumes preprocess_bank_for_fair_bbn is defined and returns the dict you specified earlier.
    print("Preprocessing dataset (bank)...")
    data_dict = preprocess_bank_for_fair_bbn('/kaggle/input/remaining2/bankmarketing/bank/bank-full.csv')
    results = train_fair_bbn_adapter_bank(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3)


Preprocessing dataset (bank)...
Training baseline MLP...


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:686: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Baseline MLP Accuracy: 0.9195
Training fair BBN...
Training adapter with adversarial + BBN fairness...
Epoch   0 | Adv Loss: 1.7091 | Adapter Loss: -0.1612
Epoch  10 | Adv Loss: 1.5219 | Adapter Loss: -0.0677
Epoch  20 | Adv Loss: 1.5220 | Adapter Loss: -0.0678
Epoch  30 | Adv Loss: 1.5221 | Adapter Loss: -0.0679
Epoch  40 | Adv Loss: 1.5221 | Adapter Loss: -0.0678
Epoch  50 | Adv Loss: 1.5221 | Adapter Loss: -0.0679
Epoch  59 | Adv Loss: 1.5220 | Adapter Loss: -0.0678

BASELINE MLP RESULTS:
 Accuracy: 0.9195
 Marital DP: 0.0932, Marital EO: 0.0467
 Job     DP: 0.4420, Job     EO: 0.3636

BBN + Adapter (Adversarial + Fairness) RESULTS:
 Accuracy: 0.5002
 Marital DP: 0.0000, Marital EO: 0.0000
 Job     DP: 0.0000, Job     EO: 0.0000
